# Playstyle Clusters — Archetype Review

Cluster players by their lifetime playstyle vector — the exact vector
`PlayerSimilarity` uses for similarity — so the dashboard can label player
archetypes. The cluster artifacts are **pipeline-owned**: `just train` refits
them from the fresh snapshot after refresh, before the similarity index
build. This notebook is the review instrument: it reuses the shared
generation (`build_cluster_artifacts` in `src.models.similarity`) and the
pipeline's configuration (`PLAYSTYLE_*` in `src.constants`) instead of
duplicating the cluster setup, so what you review here is exactly what the
pipeline produces. It regenerates from the live PostgreSQL client for
review; the next `just train` refreshes the artifacts from the fresh
snapshot.

The generation writes two runtime artifacts:

- `data/processed/cluster_assignments.parquet` — `player_id`, `cluster_id`
- `data/processed/cluster_descriptions.json` — `{cluster_id: human label}`

`PlayerSimilarity.build` consumes membership as a one-hot similarity feature
and bakes the labels into `player_metadata.json` for the directory/profile UI.

In [ ]:
from src.utils import load_env

load_env()

# ── Clustering configuration (pipeline-owned, single source of truth) ──
# n_clusters, the seed, and the reviewed archetype labels live in
# src/constants.py; the pipeline regenerates the artifacts from these on
# every train run. To rename an archetype, edit PLAYSTYLE_CLUSTER_LABELS
# there, then re-run the generation cell below (or the next `just train`).
from src.constants import (
    PLAYSTYLE_CLUSTER_LABELS,
    PLAYSTYLE_N_CLUSTERS,
    PLAYSTYLE_RANDOM_STATE,
)

n_clusters = PLAYSTYLE_N_CLUSTERS
random_state = PLAYSTYLE_RANDOM_STATE
cluster_labels = PLAYSTYLE_CLUSTER_LABELS

## Build the playstyle vectors

Query exactly the profile fields `PlayerSimilarity.build` queries, drop empty
player ids, then call `build_playstyle_matrix` — **without**
`cluster_assignments` so discovery is not skewed by a prior archetype. The
vector logic (calibrated block weighting in `src.models.similarity`) lives in
`BLOCK_WEIGHTS`/`block_slices` there; it is not duplicated here.
`build_cluster_artifacts` fits on this same matrix.

The clustering vector is five calibrated blocks — each unit-normalized (or
bounded-transformed), scaled by its explicit weight, concatenated, then
L2-normalized once — so a block's influence is its weight, never its raw
scale or dimension count:

- 4 one-hot identity columns: `handedness_L/R`, `backhand_1H/2H` (0.10)
- 13 lifetime playstyle stats (`LIFETIME_PLAYSTYLE_COLS`): serve shape and
  aggression, clutch serving, return strength (0.35)
- surface: exposure-shrunk hard/clay/grass win rates plus bounded exposure
  counts (0.25)
- reputation: bounded `current_rank`, `match_count`, `career_win_rate` (0.25)
- bio: summary-text embeddings PCA-reduced to at most 10 dims — a minor
  auxiliary block (0.05)

The primary signals are playstyle, surface, and reputation; bio is a tiny
bonus. The PCA-reduced bio coordinates cannot be
human-interpreted — no embedding coordinate maps to a label like "big
server". Cluster meaning is read from the raw profile and lifetime stats, and
from representative players, in the review cells below — never from
embedding coordinates.

In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from kneed import KneeLocator
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from umap import UMAP

from src.constants import BRONZE_PROFILES_TABLE
from src.db.client import to_dataframe
from src.models.similarity import (
    LIFETIME_PLAYSTYLE_COLS,
    build_cluster_artifacts,
    build_playstyle_matrix,
)

profiles = to_dataframe(
    f"SELECT player_id, display_name, backhand, handedness, summary FROM {BRONZE_PROFILES_TABLE}"
)
profiles = profiles[profiles["player_id"] != ""].reset_index(drop=True)
print(f"{len(profiles)} profiled players")

playstyle_matrix = build_playstyle_matrix(profiles, query=to_dataframe)
features = playstyle_matrix.to_numpy(np.float32)
print(
    f"playstyle matrix: {playstyle_matrix.shape[0]} players x {playstyle_matrix.shape[1]} features"
)

In [ ]:
# ── K sweep: inertia (elbow) + silhouette ─────────────────
k_values = range(2, 13)
inertias: dict[int, float] = {}
silhouettes: dict[int, float] = {}
# Per-K smallest/largest cluster sizes, for the K-selection summary table.
size_bounds: dict[int, tuple[int, int]] = {}
for k in k_values:
    kmeans = KMeans(n_clusters=k, n_init="auto", random_state=random_state).fit(features)
    inertias[k] = float(kmeans.inertia_)
    silhouettes[k] = float(silhouette_score(features, kmeans.labels_))
    counts = np.bincount(kmeans.labels_)
    size_bounds[k] = (int(counts.min()), int(counts.max()))

In [ ]:
# ── Elbow + silhouette plots ──────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(list(inertias), list(inertias.values()), marker="o")
axes[0].set_title("Inertia by K (elbow)")
axes[0].set_xlabel("K")
axes[0].set_ylabel("Inertia")
axes[1].plot(list(silhouettes), list(silhouettes.values()), marker="o")
axes[1].set_title("Silhouette score by K")
axes[1].set_xlabel("K")
axes[1].set_ylabel("Silhouette score")
fig.tight_layout()
plt.show()

knee = KneeLocator(list(inertias), list(inertias.values()), curve="convex", direction="decreasing")
if knee.knee is not None:
    print(
        f"KneeLocator suggests K={knee.knee}; the selection stays the explicit "
        f"n_clusters={n_clusters} above."
    )
else:
    print("KneeLocator found no clear elbow; keep the explicit n_clusters selection.")

## Choose K

The sweep cell above fitted KMeans for every K and recorded inertia,
silhouette, and the smallest/largest cluster sizes. The table below puts it
all on one screen; the Seaborn line plots make the elbow and the silhouette
peak readable at a glance. A usable K balances a high silhouette against
clusters that stay reasonably sized — tiny clusters are hard to name and
unstable across runs.

The pipeline always trains with the configured `n_clusters`; nothing here
overrides it. If the diagnostics point elsewhere, edit
`PLAYSTYLE_N_CLUSTERS` in `src/constants.py`, re-run the generation cell
below, and review the cluster profiles again.

In [ ]:
# ── K-selection summary ───────────────────────────────────
summary = pd.DataFrame(
    {
        "K": list(inertias),
        "inertia": [round(v, 1) for v in inertias.values()],
        "silhouette": [round(v, 4) for v in silhouettes.values()],
        "min_cluster_size": [size_bounds[k][0] for k in inertias],
        "max_cluster_size": [size_bounds[k][1] for k in inertias],
        "size_imbalance": [round(size_bounds[k][1] / size_bounds[k][0], 2) for k in inertias],
    }
)
print(f"K-selection summary (size_imbalance = max/min; configured n_clusters = {n_clusters}):")
print(summary.to_string(index=False))
best_k = max(silhouettes, key=lambda k: silhouettes[k])
print(
    f"\nHighest silhouette at K={best_k} ({silhouettes[best_k]:.4f}); the "
    f"pipeline uses the configured n_clusters={n_clusters}."
)

# ── Seaborn elbow + silhouette line plots ─────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.lineplot(x=list(inertias), y=list(inertias.values()), marker="o", ax=axes[0])
axes[0].set_title("Inertia by K (elbow, Seaborn)")
axes[0].set_xlabel("K")
axes[0].set_ylabel("Inertia")
sns.lineplot(x=list(silhouettes), y=list(silhouettes.values()), marker="o", ax=axes[1])
axes[1].set_title("Silhouette score by K (Seaborn)")
axes[1].set_xlabel("K")
axes[1].set_ylabel("Silhouette score")
for ax in axes:
    ax.axvline(n_clusters, color="red", linestyle="--", linewidth=1.2)
    ax.legend([f"configured n_clusters={n_clusters}"], fontsize=8)
fig.tight_layout()
plt.show()

In [ ]:
# ── Generate the runtime artifacts (fit + validate + write) and review ──
# Shared with the pipeline: deterministic KMeans on the playstyle vectors
# (no prior assignments), label-coverage validation, and the two artifact
# writes. Returns the fit so the review cells below inspect exactly what
# was written.
assignments, model = build_cluster_artifacts(
    n_clusters=n_clusters,
    labels=cluster_labels,
    query=to_dataframe,
    random_state=random_state,
)
print(f"Generated cluster artifacts from the fresh {len(profiles)}-player fit.")

# Deterministic projection (fixed seed) purely for visual review.
umap_2d = np.asarray(
    UMAP(n_components=2, random_state=random_state).fit_transform(features),
    dtype=np.float32,
)

fig, ax = plt.subplots(figsize=(10, 7))
for cluster_id in range(n_clusters):
    mask = (assignments["cluster_id"] == cluster_id).to_numpy()
    label = cluster_labels.get(str(cluster_id), f"cluster {cluster_id}")
    ax.scatter(umap_2d[mask, 0], umap_2d[mask, 1], s=12, alpha=0.8, label=label)
ax.set_title(f"2D UMAP of playstyle vectors, K={n_clusters}")
ax.legend(loc="best", fontsize=8)
plt.show()

## Review cluster profiles

Clustering runs on the full calibrated vector (identity one-hots + lifetime
playstyle stats + surface + reputation + low-weight PCA-reduced bio). The views
below therefore interpret clusters through raw stats and real players, never
embedding dimensions:

1. **Silhouette distribution** — the silhouette of every player at the fitted
   K; negative scores are likely mis-assigned players.
2. **Cluster sizes** — how many players each archetype claims; tiny clusters
   are hard to name and unstable.
3. **Centroid deviations** — each cluster's centroid minus the mean centroid,
   restricted to `LIFETIME_PLAYSTYLE_COLS` (serve shape, aggression, return
   strength, clutch). Positive = above-average for the
   archetype, negative = below.
4. **Full profile summary** — every interpretable metric averaged per cluster
   next to the overall benchmark: identity shares, lifetime stats, career
   context (match count, form, surface volume, physical attributes).
5. **Delta heatmap** — cluster average minus the overall average per lifetime
   stat (centered at 0), a quick visual read of what each archetype is above
   or below the field on.
6. **Top-ranked players** — the five highest-ranked players per cluster with
   career context and raw serve/return stats: the strongest real-world
   examples of each archetype.
7. **Nearest-centroid players** — the players closest to each centroid, the
   most representative faces of the cluster.

The labels are your call; the generation never names archetypes automatically.

In [ ]:
# ── Centroid deviations on LIFETIME_PLAYSTYLE_COLS ──────────
feature_columns = list(playstyle_matrix.columns)
lifetime_idx = [feature_columns.index(col) for col in LIFETIME_PLAYSTYLE_COLS]
centroids = model.cluster_centers_
deviation = pd.DataFrame(
    centroids[:, lifetime_idx] - centroids[:, lifetime_idx].mean(axis=0),
    columns=LIFETIME_PLAYSTYLE_COLS,
    index=[f"cluster {c} - {cluster_labels.get(str(c), '?')}" for c in range(n_clusters)],
)
print("Centroid deviations from the average playstyle (LIFETIME_PLAYSTYLE_COLS only):")
print(deviation.round(3).to_string())

# ── Nearest-centroid players per cluster ───────────────────
print("\nNearest-centroid players (most representative per cluster):")
samples_per_cluster = 5
for cluster_id in range(n_clusters):
    idx = np.where(assignments["cluster_id"] == cluster_id)[0]
    distance = np.linalg.norm(features[idx] - centroids[cluster_id], axis=1)
    nearest = idx[np.argsort(distance)[:samples_per_cluster]]
    print(f"\ncluster {cluster_id} - {cluster_labels.get(str(cluster_id), '?')}")
    for i in nearest:
        print(f"  {profiles['player_id'][i]:<14s} {profiles['display_name'][i]}")

In [ ]:
# ── Top-ranked players per cluster (real-world examples) ──
# The five highest-ranked players in each cluster, with career context and a
# compact serve/return readout. Lower rank = better; unranked players sort
# after ranked ones. This complements the nearest-centroid players above:
# those are the most representative, these are the most established.
# The merged profile frame built here is reused by the profile-summary cell
# below, so run cells top to bottom.
from src.constants import GOLD_PROFILES_TABLE


def _existing_cols(schema: str, table: str, wanted: list[str]) -> list[str]:
    cols = to_dataframe(
        "SELECT column_name FROM information_schema.columns "
        f"WHERE table_schema = '{schema}' AND table_name = '{table}'"
    )["column_name"].tolist()
    return [c for c in wanted if c in cols]


GOLD_WANTED = [
    "player_id",
    "match_count",
    "current_rank",
    "win_rate_10",
    "career_win_rate",
    "hard_matches",
    "clay_matches",
    "grass_matches",
    *LIFETIME_PLAYSTYLE_COLS,
]
BRONZE_WANTED = [
    "player_id",
    "display_name",
    "handedness",
    "backhand",
    "height",
    "turned_pro",
]
gold_cols = _existing_cols("gold", "player_profiles", GOLD_WANTED)
bronze_cols = _existing_cols("bronze", "player_profiles", BRONZE_WANTED)
cluster_profiles = assignments.merge(
    to_dataframe(f"SELECT {', '.join(gold_cols)} FROM {GOLD_PROFILES_TABLE}"),
    on="player_id",
    how="left",
).merge(
    to_dataframe(f"SELECT {', '.join(bronze_cols)} FROM {BRONZE_PROFILES_TABLE}"),
    on="player_id",
    how="left",
)

# Interpretable metric columns for the per-cluster summary: one-hot identity
# shares + lifetime stats + career context. The PCA-reduced bio block is
# excluded on purpose.
metric_cols: list[str] = []
for col, value in [("handedness", "R"), ("backhand", "2H")]:
    if col in cluster_profiles.columns:
        share_col = f"{col}_{value}_share"
        cluster_profiles[share_col] = (cluster_profiles[col] == value).astype(float)
        metric_cols.append(share_col)
for col in [
    *LIFETIME_PLAYSTYLE_COLS,
    "match_count",
    "win_rate_10",
    "career_win_rate",
    "hard_matches",
    "clay_matches",
    "grass_matches",
    "height",
    "turned_pro",
]:
    if col in cluster_profiles.columns:
        metric_cols.append(col)

PLAYER_STAT_COLS = [
    "overall_serve_points_won_pct",
    "return_points_won_pct",
    "aces_per_service_game",
    "break_point_conversion_pct",
]
show_cols = [
    *[
        c
        for c in ["display_name", "player_id", "current_rank", "match_count"]
        if c in cluster_profiles.columns
    ],
    *[c for c in ["career_win_rate", *PLAYER_STAT_COLS] if c in cluster_profiles.columns],
]
print("Top five highest-ranked players per cluster (rank: lower = better, '\u2014' = unranked):")
for cluster_id in range(n_clusters):
    top = (
        cluster_profiles[cluster_profiles["cluster_id"] == cluster_id]
        .sort_values("current_rank", na_position="last")
        .head(5)
    )
    top_table = top[show_cols].copy()
    if "current_rank" in top_table.columns:
        top_table["current_rank"] = top_table["current_rank"].apply(
            lambda r: "\u2014" if pd.isna(r) else str(int(r))
        )
    if "match_count" in top_table.columns:
        top_table["match_count"] = top_table["match_count"].astype("Int64")
    print(f"\ncluster {cluster_id} - {cluster_labels.get(str(cluster_id), '?')}")
    print(top_table.round(3).to_string(index=False))

In [ ]:
# ── Selected-K silhouette distribution ────────────────────
from sklearn.metrics import silhouette_samples

sil_scores = np.asarray(
    silhouette_samples(features, assignments["cluster_id"].to_numpy()), dtype=float
)
sil_df = pd.DataFrame(
    {
        "silhouette": sil_scores,
        "below_zero": (sil_scores < 0).astype(int),
        "cluster": [
            f"{cid} - {cluster_labels.get(str(cid), '?')}" for cid in assignments["cluster_id"]
        ],
    }
)
fig, ax = plt.subplots(figsize=(9, 4))
sns.histplot(data=sil_df, x="silhouette", hue="cluster", bins=40, alpha=0.55, ax=ax)
mean_sil = silhouettes.get(n_clusters)
if mean_sil is not None:
    ax.axvline(mean_sil, color="black", linestyle="--", linewidth=1.5)
ax.set_title(f"Silhouette distribution per player, K={n_clusters} (dashed = overall mean)")
plt.show()

per_cluster = sil_df.groupby("cluster", sort=False).agg(
    mean=("silhouette", "mean"),
    min=("silhouette", "min"),
    below_zero=("below_zero", "sum"),
)
print(
    "Per-cluster silhouette (below_zero = players with negative silhouette, likely mis-assigned):"
)
print(per_cluster.round(3).to_string())

In [ ]:
# ── Cluster sizes: count and share per archetype ──────────
counts = (
    assignments["cluster_id"].value_counts().sort_index().reindex(range(n_clusters), fill_value=0)
)
size_table = pd.DataFrame(
    {
        "cluster": [f"{c} - {cluster_labels.get(str(c), '?')}" for c in range(n_clusters)],
        "count": counts.to_numpy(),
    }
)
size_table["share"] = (size_table["count"] / size_table["count"].sum()).round(3)
print("Cluster sizes (share of profiled players):")
print(size_table.to_string(index=False))

In [ ]:
# ── Full interpretable profile summary + delta heatmap ────
# Reuses the merged `cluster_profiles` frame built in the top-ranked-players
# cell above. Averages every interpretable metric per cluster vs the overall
# benchmark: identity shares, lifetime playstyle stats, career context, and
# physical attributes. The PCA-reduced bio block is never summarized —
# there is no human label for its coordinates.
# Lifetime stats are NULL for zero-match players; mirror the vector builder
# and impute 0.0 so the cluster means describe the same numbers the fit saw.
imputed = cluster_profiles.copy()
lifetime_present = [c for c in LIFETIME_PLAYSTYLE_COLS if c in imputed.columns]
imputed[lifetime_present] = imputed[lifetime_present].fillna(0.0)
cluster_means = imputed.groupby("cluster_id")[metric_cols].mean()
overall_mean = imputed[metric_cols].mean()
labels_by_id = [f"{c} - {cluster_labels.get(str(c), '?')}" for c in cluster_means.index]

sections = {
    "Identity (one-hot shares)": ["handedness_R_share", "backhand_2H_share"],
    "Physical": ["height", "turned_pro"],
    "Serve shape": [
        "first_serve_in_pct",
        "first_serve_points_won_pct",
        "second_serve_points_won_pct",
        "overall_serve_points_won_pct",
    ],
    "Serve aggression": [
        "aces_per_first_serve",
        "aces_per_service_game",
        "double_faults_per_serve_point",
    ],
    "Clutch serving": ["break_points_saved_pct"],
    "Return strengths": [
        "return_points_won_pct",
        "first_serve_return_points_won_pct",
        "second_serve_return_points_won_pct",
        "break_point_conversion_pct",
        "break_point_opportunities_per_return_game",
    ],
    "Surface preference": ["hard_win_rate", "clay_win_rate", "grass_win_rate"],
    "Career context": [
        "match_count",
        "win_rate_10",
        "career_win_rate",
        "hard_matches",
        "clay_matches",
        "grass_matches",
    ],
}
for section, cols in sections.items():
    cols = [c for c in cols if c in metric_cols]
    if not cols:
        continue
    table = cluster_means.loc[:, cols].T
    table.columns = labels_by_id
    table["overall"] = overall_mean[cols]
    print(f"\n{section} (cluster means vs overall):")
    print(table.round(3).to_string())

heatmap_cols = [c for c in LIFETIME_PLAYSTYLE_COLS if c in cluster_means.columns]
deltas = cluster_means[heatmap_cols].sub(overall_mean[heatmap_cols], axis=1)
deltas.index = labels_by_id
fig, ax = plt.subplots(figsize=(15, max(3, 0.55 * n_clusters)))
sns.heatmap(deltas, center=0, annot=True, fmt=".2f", cmap="RdBu_r", linewidths=0.5, ax=ax)
ax.set_title(
    "Cluster mean minus overall average per lifetime stat (percentage points, centered at 0)"
)
plt.show()

## Labels

The generation cell above already fit, validated, and wrote the two runtime
artifacts with the current `PLAYSTYLE_CLUSTER_LABELS`. Validation is inside
`build_cluster_artifacts`: the label keys must exactly cover the fitted
cluster ids before anything is written, so the runtime never sees a
label-less cluster. If a cluster's provisional label needs to change (this
notebook never names archetypes automatically): edit
`PLAYSTYLE_CLUSTER_LABELS` in `src/constants.py`, then re-run the generation
cell above.

In [ ]:
# ── Next action ─────────────────────────────────────────────
print("After reviewing the cluster profiles above:")
print("  1. To rename an archetype, edit `PLAYSTYLE_CLUSTER_LABELS` in")
print("     src/constants.py and re-run the generation cell above.")
print("  2. The next index build consumes the refreshed artifacts:")
print("     - `just train` regenerates clusters from the fresh snapshot and")
print("       rebuilds the similarity index automatically (clusters after")
print("       snapshot refresh, index after clusters).")
print("     - or rebuild just the index:")
print(
    '     uv run python -c "from src.models.similarity import PlayerSimilarity; '
    'PlayerSimilarity().build()"'
)